DSO4

In [1]:
pip install transformers torch

In [2]:
"""
AVATAR DSO4: HCP Engagement Risk Detection
Simple sentiment analysis on HCP-Delegate conversation text
"""

from transformers import pipeline


# ==========================================
# 1. LOAD MODEL (once at startup)
# ==========================================

class HCPRiskAnalyzer:
    def __init__(self):
        # Pre-trained multilingual sentiment model (1-5 stars)
        self.sentiment = pipeline(
            "sentiment-analysis",
            model="nlptown/bert-base-multilingual-uncased-sentiment"
        )

        # Risk thresholds
        self.RISK_LOW = "LOW"
        self.RISK_MEDIUM = "MEDIUM"
        self.RISK_HIGH = "HIGH"

    # ==========================================
    # 2. ANALYZE SINGLE MESSAGE
    # ==========================================

    def analyze_message(self, text: str, hcp_id: str = "unknown") -> dict:
        """Analyze one HCP message for sentiment and risk"""

        # Get sentiment (1-5 stars)
        result = self.sentiment(text[:512])[0]
        stars = int(result["label"][0])

        # Determine risk and action
        if stars <= 2:
            risk = self.RISK_MEDIUM
            action = "Address concerns immediately"
        elif stars == 3:
            risk = self.RISK_HIGH
            action = "HCP indifferent - escalate to human delegate!"
        else:
            risk = self.RISK_LOW
            action = "Positive engagement - continue conversation"

        return {
            "hcp_id": hcp_id,
            "text": text[:50] + "..." if len(text) > 50 else text,
            "stars": stars,
            "sentiment": self._stars_to_label(stars),
            "risk": risk,
            "action": action,
            "confidence": round(result["score"], 3)
        }

    def _stars_to_label(self, stars: int) -> str:
        """Convert stars to text label"""
        if stars <= 2:
            return "negative"
        elif stars == 3:
            return "neutral"
        else:
            return "positive"

    # ==========================================
    # 3. ANALYZE CONVERSATION HISTORY
    # ==========================================

    def analyze_conversation(self, hcp_id: str, messages: list) -> dict:
        """
        Analyze multiple messages to detect tonal flattening
        messages: list of dicts with {"text": "...", "timestamp": "..."}
        """

        # Score all messages
        scores = []
        for msg in messages:
            result = self.sentiment(msg["text"][:512])[0]
            scores.append(int(result["label"][0]))

        # Detect pattern: negative → neutral (flattening)
        recent_avg = sum(scores[-3:]) / min(len(scores), 3) if scores else 3
        early_avg = sum(scores[:3]) / min(len(scores), 3) if len(scores) >= 3 else recent_avg

        # Flattening detection: was negative, now neutral
        flattening = (early_avg <= 2.5) and (2.8 <= recent_avg <= 3.5)

        if flattening:
            risk = "CRITICAL"
            action = "TONAL FLATTENING: HCP disengaging - urgent intervention!"
        else:
            # Use latest message risk
            risk = self._get_risk_from_score(scores[-1]) if scores else self.RISK_LOW
            action = "Monitor conversation" if risk == self.RISK_MEDIUM else "Continue normally"

        return {
            "hcp_id": hcp_id,
            "messages_count": len(messages),
            "scores": scores,
            "trend": f"{early_avg:.1f} → {recent_avg:.1f}",
            "flattening_detected": flattening,
            "risk": risk,
            "action": action
        }

    def _get_risk_from_score(self, stars: int) -> str:
        if stars <= 2:
            return self.RISK_MEDIUM
        elif stars == 3:
            return self.RISK_HIGH
        return self.RISK_LOW


# ==========================================
# 4. DEMO / TEST
# ==========================================

def main():
    # Initialize
    analyzer = HCPRiskAnalyzer()

    print("=" * 50)
    print("AVATAR DSO4: HCP Engagement Risk Detection")
    print("=" * 50)

    # Test 1: Single messages
    print("\n--- Single Message Analysis ---\n")

    test_messages = [
        ("Dr_A", "Excellent results, very satisfied!"),
        ("Dr_B", "Ce produit est trop cher et inefficace"),
        ("Dr_C", "Ok, je vais réfléchir et vous recontacter"),
        ("Dr_D", "Non merci, pas intéressé pour le moment"),
    ]

    for hcp_id, text in test_messages:
        result = analyzer.analyze_message(text, hcp_id)
        print(f"👤 {result['hcp_id']}")
        print(f"   Text: '{result['text']}'")
        print(f"   Stars: {result['stars']}/5 ({result['sentiment']})")
        print(f"   Risk: {result['risk']}")
        print(f"   Action: {result['action']}\n")

    # Test 2: Conversation history (tonal flattening)
    print("--- Conversation Trend Analysis ---\n")

    conversation = [
        {"text": "Ce prix est inacceptable!", "timestamp": "10:00"},      # 1 star
        {"text": "Je ne suis pas convaincu", "timestamp": "10:05"},      # 2 stars
        {"text": "Je vais considérer", "timestamp": "10:15"},            # 3 stars
        {"text": "Ok merci", "timestamp": "10:20"},                      # 3 stars (flattening!)
    ]

    result = analyzer.analyze_conversation("Dr_Risky", conversation)
    print(f"👤 {result['hcp_id']}")
    print(f"   Messages: {result['messages_count']}")
    print(f"   Scores: {result['scores']}")
    print(f"   Trend: {result['trend']}")
    print(f"   Flattening: {'YES ⚠️' if result['flattening_detected'] else 'No'}")
    print(f"   Risk: {result['risk']}")
    print(f"   Action: {result['action']}")


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

AVATAR DSO4: HCP Engagement Risk Detection

--- Single Message Analysis ---

👤 Dr_A
   Text: 'Excellent results, very satisfied!'
   Stars: 5/5 (positive)
   Risk: LOW
   Action: Positive engagement - continue conversation

👤 Dr_B
   Text: 'Ce produit est trop cher et inefficace'
   Stars: 1/5 (negative)
   Risk: MEDIUM
   Action: Address concerns immediately

👤 Dr_C
   Text: 'Ok, je vais réfléchir et vous recontacter'
   Stars: 3/5 (neutral)
   Risk: HIGH
   Action: HCP indifferent - escalate to human delegate!

👤 Dr_D
   Text: 'Non merci, pas intéressé pour le moment'
   Stars: 1/5 (negative)
   Risk: MEDIUM
   Action: Address concerns immediately

--- Conversation Trend Analysis ---

👤 Dr_Risky
   Messages: 4
   Scores: [1, 2, 3, 3]
   Trend: 2.0 → 2.7
   Flattening: No
   Risk: HIGH
   Action: Continue normally
